# Step 1 — Ingest Raw Customers to Bronze

**Purpose:** Land the raw customers table into Bronze **as-received**, adding only lineage
metadata. No cleaning — Bronze preserves raw fidelity (including dirty rows) for audit/replay.

### What This Notebook Does
1. Reads `shift_left_optimization.raw.customers_raw`.
2. Adds `ingestion_timestamp` and `source_file` lineage columns.
3. Writes append-only Delta with **automatic Liquid Clustering** (`CLUSTER BY AUTO`).

### Source & Target
| Direction | Table |
|-----------|-------|
| Source | `shift_left_optimization.raw.customers_raw` |
| Target (Bronze Delta) | `shift_left_optimization.bronze.customers_bronze` |

> **Prerequisites:** Run `schema_mgt/00_setup_and_data_prep` first.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RAW_TABLE    = 'shift_left_optimization.raw.customers_raw'
BRONZE_TABLE = 'shift_left_optimization.bronze.customers_bronze'

spark.sql('CREATE SCHEMA IF NOT EXISTS shift_left_optimization.bronze')
spark.sql(f'DROP TABLE IF EXISTS {BRONZE_TABLE}')

print(f'Source: {RAW_TABLE}')
print(f'Target: {BRONZE_TABLE}')

In [0]:
from pyspark.sql import functions as F

---

### Read the raw staging table

In [0]:
raw_df = spark.read.table(RAW_TABLE)

print(f'Raw rows read: {raw_df.count()}')

---

### Add lineage metadata (raw fidelity preserved)

In [0]:
bronze_df = (
    raw_df
    .withColumn('ingestion_timestamp', F.current_timestamp())
    .withColumn('source_file',         F.lit(RAW_TABLE))
)

---

### Write to Bronze (append-only) + enable automatic clustering

In [0]:
(bronze_df.write
    .format('delta')
    .mode('append')
    .saveAsTable(BRONZE_TABLE))

spark.sql(f'ALTER TABLE {BRONZE_TABLE} CLUSTER BY AUTO')
print(f'Bronze rows written: {spark.read.table(BRONZE_TABLE).count()}')

---

### Validation

In [0]:
%sql
SELECT * FROM shift_left_optimization.bronze.customers_bronze
ORDER BY customer_id
LIMIT 10

In [0]:
%sql
DESCRIBE HISTORY shift_left_optimization.bronze.customers_bronze